# DR – Druhá semestrální práce

## Zadání

1. Získejte datovou sadu průměrných denních teplot v Plzni za období od 1. ledna 1970 do 12. prosince 2025.

2. Rozdělte datovou sadu podle jednotlivých kalendářních roků.

3. Vykreslete graf průměrných ročních teplot. V samostatném grafu zobrazte průběh denních teplot v jednotlivých letech.

4. Pomocí **nepárového t-testu** zjistěte, které roky se statisticky významně liší od celkového průměru ročních teplot. Určete, které roky jsou statisticky významně chladnější a které teplejší než tento průměr. Použijte hladinu významnosti 0.01.

5. Pomocí **párového t-testu** zjistěte, které roky se statisticky významně liší od celkového průměru ročních teplot. Určete, které roky jsou statisticky významně chladnější a které teplejší než tento průměr. Použijte hladinu významnosti 0.01.

6. Vykreslete histogram odchylek a posuďte, zda jsou splněny předpoklady pro použití t-testu.

7. Použijte vhodný **párový neparametrický** test a vyhodnoťte, které roky se statisticky významně liší od celkového průměru ročních teplot. Určete, které roky jsou statisticky významně chladnější a které teplejší než tento průměr. Použijte hladinu významnosti 0.01.

8. Porovnejte výsledky nepárového a párového t-testu. Vyhodnoťte, zda se statisticky významně odlišuje více, nebo méně roků při použití nepárového či párového testu. Stejně tak porovnejte výsledky párového t-testu a zvoleného párového neparametrického testu. Pokuste se vysvětlit případné rozdíly ve výsledcích.

9. Vyhodnoťte trend průměrných ročních teplot a otestujte hypotézu, že tento trend je statisticky významně odlišný od nuly.

## Dodatečné informace z emailu
Zadání opět vypacujte ve formě ipython notebooku a zašlete mi ho.

Jako úkol nad rámec zadání si můžete širokou paletu možnosti od srovnání s jinými místy planety až po analýzu, které dny v roce se více hodí pro porovnávání teplot. V zadání je také schválně ponechána jedna (a neúmyslně možná více) nepřesnost/nevhodnost. Její ohalení a náprava také může být úkolem nad rámec zadání.

Varování: pozor na příliš časté používání API zvolené služby. Nezapomínejte vyčkat několik sekund před opakováním dotazu.

## Vypracování

### 1. Získání datavé sady

Souřadnice Plzně jsou 49° 44′ 51″ N, 13° 22′ 39″ E (zdroj: https://en.wikipedia.org/wiki/Plze%C5%88).

Následuje definice parametrů datasetu:

In [ ]:
# --------------------------------------------------
# Configuration
# --------------------------------------------------

location = {
    "name": "Pilsen",
    "continent": "Europe",
    "timezone": "Europe/Prague",
    "latitude": (49, 44, 51),
    "longitude": (13, 22, 39),
}

start_date = "1970-01-01"
end_date = "2025-12-31"

# --------------------------------------------------
# Derived parameters
# --------------------------------------------------

for key in ("latitude", "longitude"):
    degrees, minutes, seconds = location[key]
    location[key] = degrees + minutes / 60 + seconds / 3600

# --------------------------------------------------
# Output
# --------------------------------------------------

print(f"Location: {location['name']}")
print(f"Continent: {location['continent']}")
print(f"Timezone: {location['timezone']}")
print(f"Latitude: {location['latitude']:.3f}")
print(f"Longitude: {location['longitude']:.3f}")
print(f"Start date: {start_date}")
print(f"End date: {end_date}")

Následuje výpočet, kolik bodů pozorování bychom měli dostat.

In [ ]:
import datetime

date_start = datetime.date.fromisoformat(start_date)
date_end = datetime.date.fromisoformat(end_date)

n_days_expected = (date_end - date_start).days + 1

print(f"Start date: {date_start}")
print(f"End date: {date_end}")
print(f"Expected number of observations: {n_days_expected:,}")

K získání dat byla použita otavřená data s url adresou v následujícím kódu. Tento kód provádí záskání dat a kontrolu jejich správnosti.

In [ ]:
import requests
import pandas as pd

# --------------------------------------------------
# Open-Meteo API
# --------------------------------------------------

url = "https://archive-api.open-meteo.com/v1/archive"

daily_variables = [
    "temperature_2m_mean",
    "temperature_2m_min",
    "temperature_2m_max",
]

params = {
    "latitude": location["latitude"],
    "longitude": location["longitude"],
    "start_date": start_date,
    "end_date": end_date,
    "daily": daily_variables,
    "timezone": location["timezone"],
    "models": "era5",
}

# --------------------------------------------------
# Download data
# --------------------------------------------------

with requests.Session() as session:
    response = session.get(url, params=params, timeout=(10, 120))
    response.raise_for_status()
    result = response.json()

# --------------------------------------------------
# Extract data
# --------------------------------------------------

if "daily" not in result:
    raise ValueError("Daily data are missing from the API response.")

daily = result["daily"]

df = pd.DataFrame({
    "date": pd.to_datetime(daily["time"]),
    "temperature_mean": daily["temperature_2m_mean"],
    "temperature_min": daily["temperature_2m_min"],
    "temperature_max": daily["temperature_2m_max"],
})

# --------------------------------------------------
# Data validation
# --------------------------------------------------

print(f"Obtained observations: {len(df):,}")
print(f"Expected observations: {n_days_expected:,}")

if len(df) != n_days_expected:
    print("WARNING: Unexpected number of observations!")

if df["date"].duplicated().any():
    raise ValueError("Duplicate dates detected.")

if not df["date"].is_monotonic_increasing:
    raise ValueError("Dates are not in chronological order.")

missing = df.isna().sum()

if missing.any():
    print("\nMissing values:")
    print(missing[missing > 0])

# --------------------------------------------------
# API metadata
# --------------------------------------------------

print("\nTemperature units:")
print(result.get("daily_units", {}))

print(f"\nTimezone: {result.get('timezone')}")
print(f"Elevation: {result.get('elevation')} m")

df.head()

Následující kód vykreslí všechny body datové sady.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# --------------------------------------------------
# Plot daily temperature
# --------------------------------------------------

fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(
    df["date"],
    df["temperature_mean"],
    linewidth=0.4,
    alpha=0.7,
)

ax.set_title(f"Daily Mean Temperature in {location['name']}")
ax.set_ylabel("Temperature [°C]")
ax.set_xlabel("Date")

# Date formatting
ax.xaxis.set_major_locator(mdates.YearLocator(5))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

ax.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

### 2. Rozdělení datové sady

In [ ]:
# --------------------------------------------------
# Split data into annual DataFrames
# --------------------------------------------------

df["year"] = df["date"].dt.year

df_annual = {
    year: group.drop(columns="year").reset_index(drop=True)
    for year, group in df.groupby("year")
}

print(f"Number of years: {len(df_annual)}")
print(f"Available years: {list(df_annual.keys())}")

df_annual[2025]

### 3. Vykreslení průběhů teplot

Následují kód vykresluje vývoj průměrné roční teploty.

In [ ]:
# --------------------------------------------------
# Annual temperature statistics
# --------------------------------------------------

df_annual_stats = pd.DataFrame({
    year: {
        "mean": data["temperature_mean"].mean(),
    }
    for year, data in df_annual.items()
}).T
df_annual_stats.index.name = "year"

# --------------------------------------------------
# Plot annual mean temperature ± standard deviation
# --------------------------------------------------

fig, ax = plt.subplots(figsize=(14, 5))
years = df_annual_stats.index
means = df_annual_stats["mean"]
ax.plot(years, means, color="blue", linewidth=2.0)
ax.set_title(f"Annual Mean Temperature in {location['name']}")
ax.set_xlabel("Year")
ax.set_ylabel("Temperature [°C]")
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

Následující kód vykresluje průběh teplot během roku a průměrný rok.

In [ ]:
# --------------------------------------------------
# Average annual temperature profile
# --------------------------------------------------

df["day_of_year"] = df["date"].dt.dayofyear

df_average_year = (
    df.groupby("day_of_year")["temperature_mean"]
    .agg(["mean", "std"])
    .reset_index()
)

df_average_year.head()

# --------------------------------------------------
# Plot temperature profiles for all years
# --------------------------------------------------

fig, ax = plt.subplots(figsize=(14, 6))

for year, data in df_annual.items():
    ax.plot(
        data["date"].dt.dayofyear,
        data["temperature_mean"],
        color="blue",
        linewidth=0.5,
        alpha=0.15,
    )

# Average annual profile
ax.plot(
    df_average_year["day_of_year"],
    df_average_year["mean"],
    linewidth=2.0,
    color="blue",
    label="Average year",
)

ax.set_title(f"Annual Temperature Profiles in {location['name']}")
ax.set_xlabel("Day of year")
ax.set_ylabel("Temperature [°C]")

ax.set_xlim(1, 366)
ax.set_xticks(range(1, 367, 30))

ax.grid(alpha=0.3)
ax.legend()

fig.tight_layout()
plt.show()

### 4. Použití nepárových testů

Obecná formulace hypotéz:

* H0: odchylka od průměru je způsobena náhodnými poruchami.
* H1: odchyla od průměru je způsobena tím, že datová sada má jiný/nižší/vyšší průměr.

Konkrétní formulace při použití nepárového t-testu:

\begin{align}
H_0:& \dots,\\
H_1 (\text{rok je jiný}):& \dots,\\
H_1 (\text{rok je chladnější}):& \dots,\\
H_1(\text{rok je teplejší}):& \dots,\\
\end{align}

In [ ]:
tests_results = {}

In [ ]:
test_name = "t-test"
test_type = "independent"

tests_results[test_name] = {
        test_type: {
            "two-sided": {},
            "less": {},
            "greater": {},
        },
}


for year, data in df_annual.items():

    average_year = ...
    actual_year = ...

    ...
    p_value_two_sided = ...
    ...
    p_value_less = ...
    ...
    p_value_greater = ...

    tests_results[test_name][test_type]["two-sided"][year] = p_value_two_sided
    tests_results[test_name][test_type]["less"][year] = p_value_less
    tests_results[test_name][test_type]["greater"][year] = p_value_greater

Vykreslení průběhu p-hodnot pro jednotlivé roky.

In [ ]:
...

### 5. Použití párových testů

Obecná formulace hypotéz:

* H0: odchylka od průměru je způsobena náhodnými poruchami.
* H1: odchyla od průměru je způsobena tím, že datová sada má jiný/nižší/vyšší průměr.

Konkrétní formulace při použití nepárového t-testu:

\begin{align}
H_0:& \dots,\\
H_1 (\text{rok je jiný}):& \dots,\\
H_1 (\text{rok je chladnější}):& \dots,\\
H_1(\text{rok je teplejší}):& \dots,\\
\end{align}

In [ ]:
test_name = "t-test"
test_type = "paired"

tests_results[test_name] = {
        test_type: {
            "two-sided": {},
            "less": {},
            "greater": {},
        },
}


for year, data in df_annual.items():

    average_year = ...
    actual_year = ...

    ...
    p_value_two_sided = ...
    ...
    p_value_less = ...
    ...
    p_value_greater = ...

    tests_results[test_name][test_type]["two-sided"][year] = p_value_two_sided
    tests_results[test_name][test_type]["less"][year] = p_value_less
    tests_results[test_name][test_type]["greater"][year] = p_value_greater

Vykreslení průběhu p-hodnot pro jednotlivé roky.

In [ ]:
...

### 6. Analýza histogramu odchylek

Následující kód vykreslí histogram odchylek a zakreslí průběh normálního rozdělení se střední hodnotou a rozptylem odpovídající střední hodnotě a rozptylu odchylek.

In [ ]:
...

Porovnáním histogramu odchylek a odpovídajícího normálního rozdělení vidíme, že ...

### 7. Použití neparametrického párového testu

Obecná formulace hypotéz:

* H0: odchylka od průměru je způsobena náhodnými poruchami.
* H1: odchyla od průměru je způsobena tím, že datová sada má jiný/nižší/vyšší průměr.

Konkrétní formulace při použití vybraného neparametrického testu:

\begin{align}
H_0:& \dots,\\
H_1 (\text{rok je jiný}):& \dots,\\
H_1 (\text{rok je chladnější}):& \dots,\\
H_1(\text{rok je teplejší}):& \dots,\\
\end{align}

In [ ]:
test_name = ...
test_type = "paired"

tests_results[test_name] = {
        test_type: {
            "two-sided": {},
            "less": {},
            "greater": {},
        },
}


for year, data in df_annual.items():

    average_year = ...
    actual_year = ...

    ...
    p_value_two_sided = ...
    ...
    p_value_less = ...
    ...
    p_value_greater = ...

    tests_results[test_name][test_type]["two-sided"][year] = p_value_two_sided
    tests_results[test_name][test_type]["less"][year] = p_value_less
    tests_results[test_name][test_type]["greater"][year] = p_value_greater

Vykreslení průběhu p-hodnot pro jednotlivé roky.

In [ ]:
...

### 8. Analýza rozdílů použitých testů

...

Z analýzy vychází, že více/méně roků prochází nepárovým testem než párovým testem.

Za důvod tohoto rozdílu pokládám ...

...

Z analýzy vychází, že více/méně roků prochází t-testem než neparametrickým testem.

Za důvod tohoto rozdílu pokládám ...

### 9. Analýza trendu

Obecná formulace hypotézy o trendu (slop):

* H0: trend je nulový
* H1: trend je nenulový.


Nalezení trendu, vykreslení a vyhodnocení p-hodnoty.

In [ ]:
...

Z výsledků vyplývá ...

## Vypracování úkolu/ů nad rámec zadání

...

## Závěr

...